# N137 · 懒传播、动态节点与更新复合

**目标：** 正确组合懒标记并维护区间统计量。

**先修：** N136, N034, N124。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** range add/flip；push/pull；标记复合次序；区间长度；动态开点；赋值扩展。

## 从问题到算法

区间加法不需要立即更新每片叶子：节点和加上delta×长度，并保存尚未下传的增量。翻转二进制区间则把ones改为长度−ones；两次翻转抵消，标记按异或复合。只有准备访问子区间时才push，子节点变化后再pull。

## 最小实现

**本章接口：** `RangeAddSumTree`、`FlipCountTree`、`handle_sum_queries(nums1, nums2, queries)`

In [1]:
class RangeAddSumTree:
    def __init__(self,values):
        values=list(values); self.n=len(values); self.total=[0]*(4*max(1,self.n)); self.lazy=[0]*len(self.total)
        def build(node,l,r):
            if r-l==1: self.total[node]=values[l]; return
            mid=(l+r)//2; build(node*2,l,mid); build(node*2+1,mid,r); self.total[node]=self.total[node*2]+self.total[node*2+1]
        if self.n: build(1,0,self.n)
    def _apply(self,node,l,r,delta): self.total[node]+=(r-l)*delta; self.lazy[node]+=delta
    def _push(self,node,l,r):
        if self.lazy[node] and r-l>1:
            mid=(l+r)//2; tag=self.lazy[node]
            self._apply(node*2,l,mid,tag); self._apply(node*2+1,mid,r,tag); self.lazy[node]=0
    def add(self,left,right,delta):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        if left==right: return
        def update(node,l,r):
            if right<=l or r<=left: return
            if left<=l and r<=right: self._apply(node,l,r,delta); return
            self._push(node,l,r); mid=(l+r)//2; update(node*2,l,mid); update(node*2+1,mid,r)
            self.total[node]=self.total[node*2]+self.total[node*2+1]
        update(1,0,self.n)
    def query(self,left,right):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        if left==right: return 0
        def get(node,l,r):
            if right<=l or r<=left: return 0
            if left<=l and r<=right: return self.total[node]
            self._push(node,l,r); mid=(l+r)//2
            return get(node*2,l,mid)+get(node*2+1,mid,r)
        return get(1,0,self.n)

class FlipCountTree(RangeAddSumTree):
    def __init__(self,values):
        values=list(values)
        if any(x not in (0,1) for x in values): raise ValueError('binary values required')
        super().__init__(values)
    def _apply(self,node,l,r,delta):
        if delta&1: self.total[node]=r-l-self.total[node]; self.lazy[node]^=1
    def flip(self,left,right): self.add(left,right,1)
    def count(self,left,right): return self.query(left,right)

def handle_sum_queries(nums1,nums2,queries):
    tree=FlipCountTree(nums1); total=sum(nums2); out=[]
    for kind,a,b in queries:
        if kind==1: tree.flip(a,b+1)
        elif kind==2: total+=a*tree.total[1]
        elif kind==3: out.append(total)
        else: raise ValueError('unknown query type')
    return out

## 正确性、前提与复杂度

节点聚合值包含自身所有待下传更新，lazy仅表示孩子尚缺的部分，因此整段查询无需push。加法标记可相加，翻转标记按奇偶性合并。赋值后加法与加法后赋值不交换，不能把本章相加标记直接用于赋值拓展。

**代价：** 静态树建树O(n)时间空间；区间更新与查询O(log n)。查询1的输入区间闭合，转为内部半开区间；类型2、3只读根计数或总和为O(1)。这里实现静态节点；极大坐标域的动态开点与赋值标记属于拓展，不宣称已实现。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tree=RangeAddSumTree([1,2,3,4]); rows=[('初始',[tree.query(i,i+1) for i in range(4)])]
for l,r,x in [(0,4,2),(1,3,-1),(2,4,5)]:
    tree.add(l,r,x); rows.append((f'[{l},{r})加{x}',[tree.query(i,i+1) for i in range(4)]))
show_table(['操作','实际逐点值'],rows)

操作,实际逐点值
初始,"[1, 2, 3, 4]"
"[0,4)加2","[3, 4, 5, 6]"
"[1,3)加-1","[3, 3, 4, 6]"
"[2,4)加5","[3, 3, 9, 11]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
flip=FlipCountTree([1,0,1]); flip.flip(1,2); assert flip.count(0,3)==3
flip.flip(0,3); flip.flip(0,3); assert flip.count(0,3)==3
assert handle_sum_queries([1,0,1],[0,0,0],[[1,1,1],[2,1,0],[3,0,0]])==[3]
from random import Random
rng=Random(137)
for n in range(1,20):
    a=[rng.randrange(-3,4) for _ in range(n)]; bits=[rng.randrange(2) for _ in range(n)]; st=RangeAddSumTree(a); ft=FlipCountTree(bits)
    for _ in range(100):
        l=rng.randrange(n+1); r=rng.randrange(l,n+1); delta=rng.randrange(-4,5)
        st.add(l,r,delta); ft.flip(l,r)
        for i in range(l,r): a[i]+=delta; bits[i]^=1
        x=rng.randrange(n+1); y=rng.randrange(x,n+1)
        assert st.query(x,y)==sum(a[x:y]) and ft.count(x,y)==sum(bits[x:y])
assert RangeAddSumTree([]).query(0,0)==0
print('N137: 所有本章断言通过')

N137: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 加入赋值标记并说明与加法不可交换。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较离线压缩和动态开点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2569. Handling Sum Queries After Update（canonical）：[官方入口](https://leetcode.com/problems/handling-sum-queries-after-update/)
- 715. Range Module（extension）
- 732. My Calendar III（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。